In [70]:
# importing necessary libraries
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [71]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

MODEL_DIR = PROJECT_ROOT / "models"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [72]:
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Processed data directory: {PROCESSED_DATA_DIR}")

Project root: f:\dsproject\Telco_Customer_Churn
Raw data directory: f:\dsproject\Telco_Customer_Churn\data\raw
Processed data directory: f:\dsproject\Telco_Customer_Churn\data\processed


In [73]:
DATA_PATH = RAW_DATA_DIR / "telco_customer_churn.csv"

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")

Dataset shape: (7043, 21)


In [74]:
df.head()

,Unnamed: 0,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [75]:
df.rename(columns={"Unnamed: 0": "customerid"}, inplace=True)

In [76]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (7043, 21)

Columns:
['customerid', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [77]:
df.info

<bound method DataFrame.info of       customerid  gender  SeniorCitizen Partner Dependents  tenure  \
0     7590-VHVEG  Female              0     Yes         No       1   
1     5575-GNVDE    Male              0      No         No      34   
2     3668-QPYBK    Male              0      No         No       2   
3     7795-CFOCW    Male              0      No         No      45   
4     9237-HQITU  Female              0      No         No       2   
...          ...     ...            ...     ...        ...     ...   
7038  6840-RESVB    Male              0     Yes        Yes      24   
7039  2234-XADUH  Female              0     Yes        Yes      72   
7040  4801-JZAZL  Female              0     Yes        Yes      11   
7041  8361-LTMKD    Male              1     Yes         No       4   
7042  3186-AJIEK    Male              0      No         No      66   

     PhoneService     MultipleLines InternetService OnlineSecurity  ...  \
0              No  No phone service             DSL 

In [78]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customerid,7043,7043,7590-VHVEG,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,7043,2,Male,3555,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SeniorCitizen,7043.0,NaN,NaN,NaN,0.162147,0.368612,0.0,0.0,0.0,0.0,1.0
Partner,7043,2,No,3641,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dependents,7043,2,No,4933,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure,7043.0,NaN,NaN,NaN,32.371149,24.559481,0.0,9.0,29.0,55.0,72.0
PhoneService,7043,2,Yes,6361,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MultipleLines,7043,3,No,3390,NaN,NaN,NaN,NaN,NaN,NaN,NaN
InternetService,7043,3,Fiber optic,3096,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OnlineSecurity,7043,3,No,3498,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [79]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [80]:
df.columns.tolist()

['customerid',
 'gender',
 'seniorcitizen',
 'partner',
 'dependents',
 'tenure',
 'phoneservice',
 'multiplelines',
 'internetservice',
 'onlinesecurity',
 'onlinebackup',
 'deviceprotection',
 'techsupport',
 'streamingtv',
 'streamingmovies',
 'contract',
 'paperlessbilling',
 'paymentmethod',
 'monthlycharges',
 'totalcharges',
 'churn']

In [81]:
object_columns = df.select_dtypes(include="object").columns

for column in object_columns:
    df[column] = df[column].str.strip()

In [82]:
for column in object_columns:
    print(f"{column}: {df[column].unique()[:10]}")

customerid: ['7590-VHVEG' '5575-GNVDE' '3668-QPYBK' '7795-CFOCW' '9237-HQITU'
 '9305-CDSKC' '1452-KIOVK' '6713-OKOMC' '7892-POOKP' '6388-TABGU']
gender: ['Female' 'Male']
partner: ['Yes' 'No']
dependents: ['No' 'Yes']
phoneservice: ['No' 'Yes']
multiplelines: ['No phone service' 'No' 'Yes']
internetservice: ['DSL' 'Fiber optic' 'No']
onlinesecurity: ['No' 'Yes' 'No internet service']
onlinebackup: ['Yes' 'No' 'No internet service']
deviceprotection: ['No' 'Yes' 'No internet service']
techsupport: ['No' 'Yes' 'No internet service']
streamingtv: ['No' 'Yes' 'No internet service']
streamingmovies: ['No' 'Yes' 'No internet service']
contract: ['Month-to-month' 'One year' 'Two year']
paperlessbilling: ['Yes' 'No']
paymentmethod: ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']
totalcharges: ['29.85' '1889.5' '108.15' '1840.75' '151.65' '820.5' '1949.4' '301.9'
 '3046.05' '3487.95']
churn: ['No' 'Yes']


In [83]:
df["totalcharges"].dtype

dtype('O')

In [84]:
df["totalcharges"] = pd.to_numeric(
    df["totalcharges"],
    errors="coerce"
)

In [85]:
df["totalcharges"].dtype

dtype('float64')

In [86]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": df.isna().mean() * 100
})

missing = missing[
    missing["missing_count"] > 0
].sort_values(
    "missing_count",
    ascending=False
)

missing

,missing_count,missing_percentage
totalcharges,11,0.156183


In [87]:
df[df["totalcharges"].isna()][
    ["customerid", "tenure", "monthlycharges", "totalcharges"]
]

,customerid,tenure,monthlycharges,totalcharges
488,4472-LVYGI,0,52.55,NaN
753,3115-CZMZD,0,20.25,NaN
936,5709-LVOEQ,0,80.85,NaN
1082,4367-NUYAO,0,25.75,NaN
1340,1371-DWPAZ,0,56.05,NaN
3331,7644-OMVMY,0,19.85,NaN
3826,3213-VVOLG,0,25.35,NaN
4380,2520-SGTTA,0,20.00,NaN
5218,2923-ARZLG,0,19.70,NaN
6670,4075-WKNIU,0,73.35,NaN


In [88]:
print(
    f"Missing TotalCharges: "
    f"{df['totalcharges'].isna().sum()}"
)

Missing TotalCharges: 11


In [89]:
df["churn"].value_counts(dropna=False)

churn
No     5174
Yes    1869
Name: count, dtype: int64

In [90]:
df['churn'].unique()

array(['No', 'Yes'], dtype=object)

In [91]:
expected_churn_values = {"Yes", "No"}

actual_churn_values = set(df["churn"].dropna().unique())

unexpected_values = actual_churn_values - expected_churn_values

print("Unexpected churn values:", unexpected_values)

Unexpected churn values: set()


In [92]:
print("Tenure minimum:", df["tenure"].min())
print("Tenure maximum:", df["tenure"].max())

Tenure minimum: 0
Tenure maximum: 72


In [93]:
df[
    (df["tenure"] < 0)
]

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn


In [94]:
print("MonthlyCharges minimum:", df["monthlycharges"].min())
print("MonthlyCharges maximum:", df["monthlycharges"].max())

MonthlyCharges minimum: 18.25
MonthlyCharges maximum: 118.75


In [95]:
df['seniorcitizen'].value_counts(dropna=False)

seniorcitizen
0    5901
1    1142
Name: count, dtype: int64

In [96]:
invalid_senior_citizen = df[
    ~df["seniorcitizen"].isin([0, 1])
]

invalid_senior_citizen

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn


In [97]:
invalid_senior_citizen = df[
    ~df["seniorcitizen"].isin([0, 1])
]

invalid_senior_citizen

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn


In [98]:
duplicate_rows = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_rows}")

Duplicate rows: 0


In [99]:
duplicate_customer_ids = df["customerid"].duplicated().sum()

print(
    f"Duplicate customer IDs: "
    f"{duplicate_customer_ids}"
)

Duplicate customer IDs: 0


In [100]:
df["customerid"].nunique(), len(df)

(7043, 7043)

In [101]:
df = df.drop(columns=["customerid"])

In [102]:
"customerid" in df.columns

False

In [103]:
# Convert "no" to 1 and "yes" to 0

df["churn"] = df["churn"].map({
    "No": 0,
    "Yes": 1
}) 

In [104]:
df["churn"].value_counts()

churn
0    5174
1    1869
Name: count, dtype: int64

In [105]:
df['churn'].dtype

dtype('int64')

In [106]:
assert df["churn"].notna().all(), \
    "Target contains missing values."

assert set(df["churn"].unique()).issubset({0, 1}), \
    "Unexpected target values."

In [107]:
print("Final shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum())

print("\nTarget distribution:")
print(df["churn"].value_counts())

Final shape: (7043, 20)

Missing values:
gender               0
seniorcitizen        0
partner              0
dependents           0
tenure               0
phoneservice         0
multiplelines        0
internetservice      0
onlinesecurity       0
onlinebackup         0
deviceprotection     0
techsupport          0
streamingtv          0
streamingmovies      0
contract             0
paperlessbilling     0
paymentmethod        0
monthlycharges       0
totalcharges        11
churn                0
dtype: int64

Target distribution:
churn
0    5174
1    1869
Name: count, dtype: int64


In [108]:
INTERIM_DATA_DIR = DATA_DIR / "interim"

INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)

clean_data_path = INTERIM_DATA_DIR / "clean_data.csv"

df.to_csv(clean_data_path, index=False)

print(f"Clean data saved to: {clean_data_path}")
print(f"Clean data shape: {df.shape}")

Clean data saved to: f:\dsproject\Telco_Customer_Churn\data\interim\clean_data.csv
Clean data shape: (7043, 20)


In [109]:
df = pd.read_csv(clean_data_path)

print(f"Loaded clean data shape: {df.shape}")
df.head()

Loaded clean data shape: (7043, 20)


,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,onlinebackup,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [110]:
X = df.drop(columns=["churn"])
y = df["churn"]

In [111]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7043, 19)
y shape: (7043,)


In [112]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

In [113]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

In [114]:
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Training: (4930, 19)
Validation: (1056, 19)
Test: (1057, 19)


In [115]:
# Verifying target distribution
print("Overall:")
print(y.value_counts(normalize=True))

print("\nTraining:")
print(y_train.value_counts(normalize=True))

print("\nValidation:")
print(y_val.value_counts(normalize=True))

print("\nTest:")
print(y_test.value_counts(normalize=True))

Overall:
churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64

Training:
churn
0    0.734686
1    0.265314
Name: proportion, dtype: float64

Validation:
churn
0    0.734848
1    0.265152
Name: proportion, dtype: float64

Test:
churn
0    0.734153
1    0.265847
Name: proportion, dtype: float64


In [116]:
# Identifying numerical features
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

numerical_features

['seniorcitizen', 'tenure', 'monthlycharges', 'totalcharges']

In [117]:
# Identifying categorical features
categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

categorical_features

['gender',
 'partner',
 'dependents',
 'phoneservice',
 'multiplelines',
 'internetservice',
 'onlinesecurity',
 'onlinebackup',
 'deviceprotection',
 'techsupport',
 'streamingtv',
 'streamingmovies',
 'contract',
 'paperlessbilling',
 'paymentmethod']

## Build the Numerical Preprocessing Pipeline


Numerical features are processed in two sequential steps:

| Step | Transformer | Rationale |
|---|---|---|
| 1. Imputation | `SimpleImputer(strategy="median")` | Median is robust to outliers, unlike the mean |
| 2. Scaling | `StandardScaler` | Standardizes features to zero mean and unit variance, which benefits distance- and gradient-based models |


In [118]:
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

## Build the Categorical Preprocessing Pipeline

Categorical features are processed in two sequential steps:

| Step | Transformer | Rationale |
|---|---|---|
| 1. Imputation | `SimpleImputer(strategy="most_frequent")` | Fills missing values with the mode, preserving category validity |
| 2. Encoding | `OneHotEncoder(handle_unknown="ignore")` | Avoids ordinal assumptions; unseen categories at inference time are encoded as all-zeros instead of raising an error |


In [119]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

## Combine Pipelines with ColumnTransformer

Both pipelines are applied in parallel via `ColumnTransformer`. Columns not listed in either transformer are dropped (`remainder="drop"`), ensuring the output contains only explicitly engineered features.


In [120]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder="drop"
)

### Preprocessing Pipeline Architecture

The `ColumnTransformer` applies the numerical and categorical pipelines in parallel and concatenates their outputs into a single transformed feature matrix:

```mermaid
flowchart TD
    X["Input Features (X)"] --> CT{"ColumnTransformer"}
    CT --> NUM["Numerical Pipeline"]
    CT --> CAT["Categorical Pipeline"]
    NUM --> NI["Median Imputer"]
    NI --> NS["Standard Scaler"]
    CAT --> CI["Most-Frequent Imputer"]
    CI --> CE["One-Hot Encoder<br/>(handle_unknown='ignore')"]
    NS --> OUT["Transformed Feature Matrix"]
    CE --> OUT
```

| Branch | Steps | Rationale |
|---|---|---|
| **Numerical** | Median imputation → standard scaling | Median is robust to outliers; scaling normalizes feature ranges |
| **Categorical** | Mode imputation → one-hot encoding | Unknown categories at inference time are safely ignored |


# Fitting preprocessing only on training data 

In [121]:
X_train_processed = preprocessor.fit_transform(X_train)

In [122]:
X_val_processed = preprocessor.transform(X_val)

X_test_processed = preprocessor.transform(X_test)

# Processed shapes

In [ ]:
print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

print("Original validation shape:", X_val.shape)
print("Processed validation shape:", X_val_processed.shape)

print("Original test shape:", X_test.shape)
print("Processed test shape:", X_test_processed.shape)

Original training shape: (4930, 19)
Processed training shape: (4930, 45)
Original validation shape: (1056, 19)
Processed validation shape: (1056, 45)
Original test shape: (1057, 19)
Processed test shape: (1057, 45)


# Checking for missing values in processed datasets

In [ ]:
print(
    "Training missing values:",
    np.isnan(X_train_processed).sum()
)

print(
    "Validation missing values:",
    np.isnan(X_val_processed).sum()
)

print(
    "Test missing values:",
    np.isnan(X_test_processed).sum()
)

Training missing values: 0
Validation missing values: 0
Test missing values: 0


In [ ]:
# Inspecting generated feature names
feature_names = preprocessor.get_feature_names_out()

print(f"Number of features: {len(feature_names)}")

Number of features: 45


In [ ]:
feature_names[:10]

array(['numerical__seniorcitizen', 'numerical__tenure',
       'numerical__monthlycharges', 'numerical__totalcharges',
       'categorical__gender_Female', 'categorical__gender_Male',
       'categorical__partner_No', 'categorical__partner_Yes',
       'categorical__dependents_No', 'categorical__dependents_Yes'],
      dtype=object)

In [ ]:
# Convert processed data into DataFrames
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_val_processed_df = pd.DataFrame(
    X_val_processed,
    columns=feature_names,
    index=X_val.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

In [ ]:
# Displaying the first few rows of the processed training DataFrame
X_train_processed_df.head()

,numerical__seniorcitizen,numerical__tenure,numerical__monthlycharges,numerical__totalcharges,categorical__gender_Female,categorical__gender_Male,categorical__partner_No,categorical__partner_Yes,categorical__dependents_No,categorical__dependents_Yes,...,categorical__streamingmovies_Yes,categorical__contract_Month-to-month,categorical__contract_One year,categorical__contract_Two year,categorical__paperlessbilling_No,categorical__paperlessbilling_Yes,categorical__paymentmethod_Bank transfer (automatic),categorical__paymentmethod_Credit card (automatic),categorical__paymentmethod_Electronic check,categorical__paymentmethod_Mailed check
5557,-0.438147,-1.114728,0.504286,-0.837871,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2270,2.282338,-1.195884,0.724189,-0.909151,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
6930,-0.438147,-1.195884,0.337292,-0.910985,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
2257,-0.438147,1.117066,0.515860,1.110142,1.0,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
898,-0.438147,-0.830682,1.122660,-0.516301,1.0,0.0,1.0,0.0,1.0,0.0,...,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0


In [ ]:
# Verifying sclaing
X_train_processed_df.filter(
    like="numerical__"
).describe().T

,count,mean,std,min,25%,50%,75%,max
numerical__seniorcitizen,4930.0,-6.773937e-17,1.000101,-0.438147,-0.438147,-0.438147,-0.438147,2.282338
numerical__tenure,4930.0,1.167423e-16,1.000101,-1.317619,-0.952416,-0.140854,0.954754,1.604003
numerical__monthlycharges,4930.0,4.143632e-17,1.000101,-1.539324,-0.973859,0.186005,0.830007,1.779062
numerical__totalcharges,4930.0,1.232280e-16,1.000101,-0.997368,-0.830472,-0.399210,0.677991,2.785321


In [ ]:
# Verifying categorical encoding
X_train_processed_df.filter(
    like="categorical__"
).head()

,categorical__gender_Female,categorical__gender_Male,categorical__partner_No,categorical__partner_Yes,categorical__dependents_No,categorical__dependents_Yes,categorical__phoneservice_No,categorical__phoneservice_Yes,categorical__multiplelines_No,categorical__multiplelines_No phone service,...,categorical__streamingmovies_Yes,categorical__contract_Month-to-month,categorical__contract_One year,categorical__contract_Two year,categorical__paperlessbilling_No,categorical__paperlessbilling_Yes,categorical__paymentmethod_Bank transfer (automatic),categorical__paymentmethod_Credit card (automatic),categorical__paymentmethod_Electronic check,categorical__paymentmethod_Mailed check
5557,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2270,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
6930,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
2257,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
898,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0


In [ ]:
# validating the preprocessing pipeline
assert X_train_processed.shape[0] == X_train.shape[0]
assert X_val_processed.shape[0] == X_val.shape[0]
assert X_test_processed.shape[0] == X_test.shape[0]

assert X_train_processed.shape[1] == X_val_processed.shape[1]
assert X_train_processed.shape[1] == X_test_processed.shape[1]

In [ ]:
assert not np.isnan(X_train_processed).any()
assert not np.isnan(X_val_processed).any()
assert not np.isnan(X_test_processed).any()

In [ ]:
# saving the preprocessing pipeline
PREPROCESSOR_PATH = MODEL_DIR / "preprocessor.joblib"

joblib.dump(
    preprocessor,
    PREPROCESSOR_PATH
)

print(f"Preprocessor saved to: {PREPROCESSOR_PATH}")

Preprocessor saved to: f:\dsproject\Telco_Customer_Churn\models\preprocessor.joblib


In [ ]:
# Load the saved preprocessing pipeline
preprocessor = joblib.load(
    MODEL_DIR / "preprocessor.joblib"
)


In [ ]:
# save the processed dataset
# X
X_train_processed_df.to_csv(
    PROCESSED_DATA_DIR / "X_train.csv",
    index=False
)

X_val_processed_df.to_csv(
    PROCESSED_DATA_DIR / "X_validation.csv",
    index=False
)

X_test_processed_df.to_csv(
    PROCESSED_DATA_DIR / "X_test.csv",
    index=False
)

In [ ]:
# Save target
y_train.to_csv(
    PROCESSED_DATA_DIR / "y_train.csv",
    index=False
)

y_val.to_csv(
    PROCESSED_DATA_DIR / "y_validation.csv",
    index=False
)

y_test.to_csv(
    PROCESSED_DATA_DIR / "y_test.csv",
    index=False
)

In [ ]:
# Save te feature names
FEATURE_NAMES_PATH = PROCESSED_DATA_DIR / "feature_names.txt"

with open(FEATURE_NAMES_PATH, "w") as file:
    for feature in feature_names:
        file.write(f"{feature}\n")

In [ ]:
# preprocessing summary
preprocessing_summary = {
    "original_rows": len(df),
    "original_features": X.shape[1],
    "training_rows": X_train.shape[0],
    "validation_rows": X_val.shape[0],
    "test_rows": X_test.shape[0],
    "processed_features": X_train_processed.shape[1],
    "numerical_features": len(numerical_features),
    "categorical_features": len(categorical_features),
    "preprocessor": str(PREPROCESSOR_PATH)
}

preprocessing_summary

{'original_rows': 7043,
 'original_features': 19,
 'training_rows': 4930,
 'validation_rows': 1056,
 'test_rows': 1057,
 'processed_features': 45,
 'numerical_features': 4,
 'categorical_features': 15,
 'preprocessor': 'f:\\dsproject\\Telco_Customer_Churn\\models\\preprocessor.joblib'}

## Preprocessing Decisions

### Data cleaning

- Standardized column names.
- Removed leading/trailing whitespace from categorical values.
- Converted `TotalCharges` from string to numeric.
- Invalid numeric values were converted to missing values for controlled imputation.
- Validated duplicate records and customer IDs.

### Feature handling

- Removed `customerid` because it is an identifier and has no meaningful predictive interpretation.
- Converted the `churn` target from `Yes/No` to `1/0`.

### Numerical preprocessing

- Missing numerical values are imputed using the median.
- Numerical variables are standardized using `StandardScaler`.

### Categorical preprocessing

- Missing categorical values are imputed using the most frequent category.
- Categorical variables are transformed using one-hot encoding.
- `handle_unknown="ignore"` is used to safely handle unseen categories.

### Data splitting

The dataset was divided into:

- 70% training
- 15% validation
- 15% test

Stratified splitting was used to preserve the churn class distribution.

### Leakage prevention

The preprocessing pipeline is fitted exclusively on the training dataset.

Validation and test datasets are transformed using the already-fitted training pipeline.

### Output

The preprocessing pipeline and processed datasets are saved for downstream model development.